# Synergy-AMP: leakage-controlled benchmark of AMP–antibiotic synergy prediction

**Question.** Do peptide sequence features (ESM-2 embeddings, simple physicochemical numbers) improve prediction of
antimicrobial-peptide/antibiotic synergy beyond knowing *which antibiotic* is used *against which bacterium*?

**Data.** `synergy_clean.csv`, built by `scripts/02_clean_dataset.py` from DBAASP (2,325 rows, 419 peptides, 86 antibiotics, 22 referenced studies).
The data file is not redistributed in this repository; see the README for how to rebuild it.

**How to run.** Colab → Runtime → Change runtime type → **T4 GPU** → Runtime → **Restart and run all**.
When the upload cell asks, choose `synergy_clean.csv`. Total run time is roughly 15–30 minutes (the leave-one-out tests are the slow part).

**Sections:** 1 setup · 2 load · 3 features · 4 peptide-family groups · 5 scoring · 6 the three fair tests · 7 five-seed comparison · 8 robustness checks · 9 what drives the baseline

## 1. Setup

In [ ]:
!pip install -q transformers rdkit xgboost biopython scikit-learn scipy shap
from google.colab import files
uploaded = files.upload()    # choose synergy_clean.csv (assigning it stops Colab echoing the file)

Saving synergy_clean.csv to synergy_clean.csv


## 2. Load the clean table

In [ ]:
import numpy as np, pandas as pd, torch, xgboost as xgb
from transformers import AutoTokenizer, EsmModel
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Align import PairwiseAligner
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef

df = pd.read_csv("synergy_clean.csv").reset_index(drop=True)
y = df["label"].values
print("rows:", len(df), "| peptides:", df["sequence"].nunique(),
      "| synergy rate:", y.mean().round(3))
one_paper = (df.groupby("sequence")["reference"].nunique() == 1).mean()
print("share of peptides seen in only one paper:", round(float(one_paper), 2))

rows: 2325 | peptides: 419 | synergy rate: 0.467
share of peptides seen in only one paper: 0.96


## 3. Features
Every row becomes a list of numbers: peptide (320 ESM-2 + 7 simple properties), antibiotic (5 RDKit descriptors + 256-bit Morgan fingerprint),
bacterium (Gram-negative flag + one-hot species). `X_base` deliberately contains **no peptide information**; it is the baseline we must beat.

In [ ]:
# ---------- peptide features ----------
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D").to(dev).eval()

@torch.no_grad()
def esm_embed(seqs, bs=32):
    out = []
    for i in range(0, len(seqs), bs):
        b = tok(seqs[i:i + bs], return_tensors="pt", padding=True).to(dev)
        h = esm(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out)                      # one 320-number vector per peptide

EISEN = dict(A=.25, R=-1.76, N=-.64, D=-.72, C=.04, Q=-.69, E=-.62, G=.16, H=-.40, I=.73,
             L=.53, K=-1.10, M=.26, F=.61, P=-.07, S=-.26, T=-.18, W=.37, Y=.02, V=.54)

def hmoment(seq, ang=100):
    a = np.deg2rad(ang) * np.arange(len(seq))
    h = np.array([EISEN[c] for c in seq])
    return np.hypot((h * np.cos(a)).sum(), (h * np.sin(a)).sum()) / len(seq)

def pep_phys(s):
    p = ProteinAnalysis(s)
    return [len(s), p.charge_at_pH(7.4), p.gravy(), p.isoelectric_point(),
            p.aromaticity(), hmoment(s), sum(s.count(a) for a in "KR") / len(s)]

peps = df["sequence"].unique().tolist()
emb = dict(zip(peps, esm_embed(peps)))
phys = {s: pep_phys(s) for s in peps}

# ---------- antibiotic features ----------
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)

def abx_feat(smi):
    m = Chem.MolFromSmiles(smi)
    d = [Descriptors.MolLogP(m), Descriptors.MolWt(m), Descriptors.TPSA(m),
         Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)]
    return np.concatenate([d, fpgen.GetFingerprintAsNumPy(m).astype(float)])

abx = {s: abx_feat(s) for s in df["smiles"].unique()}

# ---------- bacterium features ----------
species_cols = list(pd.get_dummies(df["species"]).columns)

X_esm = np.vstack([emb[s] for s in df["sequence"]])
X_pep = np.vstack([phys[s] for s in df["sequence"]])
X_abx = np.vstack([abx[s] for s in df["smiles"]])
X_bug = np.hstack([df[["gram_neg"]].values,
                   pd.get_dummies(df["species"]).values.astype(float)])

X_full = np.hstack([X_esm, X_pep, X_abx, X_bug])    # everything
X_base = np.hstack([X_abx, X_bug])                  # NO peptide information
X_phys = np.hstack([X_pep, X_abx, X_bug])           # simple peptide numbers only
X_esmab = np.hstack([X_esm, X_abx, X_bug])          # ESM peptide vector only
X_peponly = np.hstack([X_esm, X_pep])               # ONLY peptide info (control)
print("feature matrix:", X_full.shape)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


feature matrix: (2325, 627)


## 4. Group similar peptides
Similar peptides must never sit on both sides of a train/test split. Similarity = global alignment with gap penalties; clustering = average linkage.
A first attempt (longest-common-subsequence similarity, single-linkage chaining) put 82 % of all rows into one group and made the grouped test meaningless; this version keeps the biggest group near 10 %.

In [ ]:
# peptide similarity = global alignment with gap penalties, as a fraction of the longer peptide
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score, aligner.mismatch_score = 1, 0
aligner.open_gap_score, aligner.extend_gap_score = -1, -0.5

def pep_sim(a, b):
    return max(aligner.score(a, b), 0) / max(len(a), len(b))

n = len(peps)
D = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        D[i, j] = D[j, i] = 1 - pep_sim(peps[i], peps[j])
Z = linkage(squareform(D, checks=False), method="average")   # average linkage: no chaining

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    lab = fcluster(Z, t=t, criterion="distance")
    share = df["sequence"].map(dict(zip(peps, lab))).value_counts().iloc[0] / len(df)
    print(f"distance<{t}: {len(set(lab))} clusters | biggest = {share:.0%} of rows")

T = 0.8      # chosen: loosest setting with no dominant group
lab = fcluster(Z, t=T, criterion="distance")
g_pep = df["sequence"].map(dict(zip(peps, lab))).factorize()[0]     # peptide-family groups
g_abx = df["antibiotic"].factorize()[0]                             # one group per antibiotic
g_ref = df["reference"].factorize()[0]       # one group per paper (missing = its own group)
print("using T =", T, "->", len(set(g_pep)), "peptide groups | biggest:",
      pd.Series(g_pep).value_counts().max(), "rows")

distance<0.5: 162 clusters | biggest = 5% of rows
distance<0.6: 144 clusters | biggest = 5% of rows
distance<0.7: 117 clusters | biggest = 9% of rows
distance<0.8: 71 clusters | biggest = 10% of rows
distance<0.9: 28 clusters | biggest = 30% of rows
using T = 0.8 -> 71 peptide groups | biggest: 237 rows


## 5. Scoring machinery

In [ ]:
def make(seed=0):
    return xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.5, eval_metric="logloss",
                             n_jobs=-1, random_state=seed)

def evaluate(X, groups, cv, mask=None, seed=0):
    """Out-of-group predictions for every row, then pooled scores."""
    Xm, ym, gm = X, y, groups
    if mask is not None:
        Xm, ym, gm = X[mask], y[mask], groups[mask]
    oof = np.zeros(len(ym))
    for tr, te in cv.split(Xm, ym, gm):
        oof[te] = make(seed).fit(Xm[tr], ym[tr]).predict_proba(Xm[te])[:, 1]
    return {"AUPRC": round(float(average_precision_score(ym, oof)), 3),
            "AUROC": round(float(roc_auc_score(ym, oof)), 3),
            "MCC": round(float(matthews_corrcoef(ym, oof > 0.5)), 3)}

## 6. The three fair tests
1. **New peptide families**: whole peptide groups are held out.
2. **New antibiotic**: all rows of one antibiotic are held out.
3. **New paper**: all rows from one study are held out.

Each test is compared with the no-peptide **baseline**, and with a **peptide-only** control to detect memorization.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
print("prevalence (a coin flip scores AUPRC about this):", y.mean().round(3))

print("\nTEST 1: NEW PEPTIDE FAMILIES (peptide-group split, seed 0)")
print("BASELINE (antibiotic+bug)  :", evaluate(X_base, g_pep, sgkf))
print("PHYSCHEM peptide           :", evaluate(X_phys, g_pep, sgkf))
print("ESM peptide                :", evaluate(X_esmab, g_pep, sgkf))
print("FULL                       :", evaluate(X_full, g_pep, sgkf))

print("\nTEST 2: NEW ANTIBIOTIC (leave-one-antibiotic-out)")
print("FULL                       :", evaluate(X_full, g_abx, LeaveOneGroupOut()))
print("BASELINE                   :", evaluate(X_base, g_abx, LeaveOneGroupOut()))
print("PEPTIDE-ONLY (memorizing?) :", evaluate(X_peponly, g_abx, LeaveOneGroupOut()))

print("\nTEST 3: NEW PAPER (leave-one-paper-out)")
print("FULL                       :", evaluate(X_full, g_ref, LeaveOneGroupOut()))
print("BASELINE                   :", evaluate(X_base, g_ref, LeaveOneGroupOut()))
print("PEPTIDE-ONLY               :", evaluate(X_peponly, g_ref, LeaveOneGroupOut()))

prevalence (a coin flip scores AUPRC about this): 0.467

TEST 1: NEW PEPTIDE FAMILIES (peptide-group split, seed 0)
BASELINE (antibiotic+bug)  : {'AUPRC': 0.598, 'AUROC': 0.647, 'MCC': 0.195}
PHYSCHEM peptide           : {'AUPRC': 0.555, 'AUROC': 0.581, 'MCC': 0.089}
ESM peptide                : {'AUPRC': 0.532, 'AUROC': 0.585, 'MCC': 0.145}
FULL                       : {'AUPRC': 0.531, 'AUROC': 0.594, 'MCC': 0.146}

TEST 2: NEW ANTIBIOTIC (leave-one-antibiotic-out)
FULL                       : {'AUPRC': 0.743, 'AUROC': 0.758, 'MCC': 0.376}
BASELINE                   : {'AUPRC': 0.552, 'AUROC': 0.59, 'MCC': 0.114}
PEPTIDE-ONLY (memorizing?) : {'AUPRC': 0.757, 'AUROC': 0.767, 'MCC': 0.384}

TEST 3: NEW PAPER (leave-one-paper-out)
FULL                       : {'AUPRC': 0.635, 'AUROC': 0.652, 'MCC': 0.236}
BASELINE                   : {'AUPRC': 0.645, 'AUROC': 0.669, 'MCC': 0.247}
PEPTIDE-ONLY               : {'AUPRC': 0.588, 'AUROC': 0.613, 'MCC': 0.171}


## 7. Five-seed comparison on new peptide families
With only ~70 peptide groups a single split can be lucky or unlucky, so each model is scored on 5 different splits (mean ± std).
The list of variants was fixed in advance.

In [ ]:
# 5 random splits per model, so we can see the spread
X_pca10 = PCA(n_components=10, random_state=0).fit_transform(X_esm)   # uses no labels
variants = {
    "antibiotic only":               X_abx,
    "bacterium only":                X_bug,
    "BASELINE (antibiotic+bug)":     X_base,
    "BASE + charge,hmoment,length":  np.hstack([X_pep[:, [1, 5, 0]], X_base]),
    "BASE + all 7 simple peptide":   X_phys,
    "BASE + ESM reduced to 10":      np.hstack([X_pca10, X_base]),
    "FULL":                          X_full,
}
for name, X in variants.items():
    res = []
    for sd in range(5):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=sd)
        res.append(evaluate(X, g_pep, cv, seed=sd))
    out = "  ".join(f"{m}: {np.mean([r[m] for r in res]):.3f}+/-{np.std([r[m] for r in res]):.3f}"
                    for m in ["AUROC", "AUPRC", "MCC"])
    print(f"{name:30s} {out}")

antibiotic only                AUROC: 0.594+/-0.009  AUPRC: 0.541+/-0.011  MCC: 0.122+/-0.016
bacterium only                 AUROC: 0.506+/-0.025  AUPRC: 0.507+/-0.016  MCC: 0.002+/-0.064
BASELINE (antibiotic+bug)      AUROC: 0.643+/-0.005  AUPRC: 0.593+/-0.009  MCC: 0.207+/-0.014
BASE + charge,hmoment,length   AUROC: 0.613+/-0.016  AUPRC: 0.572+/-0.015  MCC: 0.167+/-0.028
BASE + all 7 simple peptide    AUROC: 0.590+/-0.024  AUPRC: 0.562+/-0.019  MCC: 0.110+/-0.036
BASE + ESM reduced to 10       AUROC: 0.592+/-0.017  AUPRC: 0.548+/-0.014  MCC: 0.125+/-0.023
FULL                           AUROC: 0.586+/-0.010  AUPRC: 0.533+/-0.016  MCC: 0.127+/-0.014


## 8. Robustness checks
* **Polymyxin-B stand-in.** PubChem returned no structure for polymyxin B, so rows for it use the colistin structure (`proxy = 1`). Does removing them change anything?
* **Paper effects.** How much of the data comes from each study?

In [ ]:
# 1. does the polymyxin-B stand-in (colistin structure) matter? (5 seeds)
mask = (df["proxy"] == 0).values
for name, X in {"BASELINE": X_base, "FULL": X_full}.items():
    for tag, m in [("all rows", None), ("without polymyxin-B rows", mask)]:
        res = [evaluate(X, g_pep, StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=sd),
                        mask=m, seed=sd) for sd in range(5)]
        a = [r["AUROC"] for r in res]
        print(f"{name:9s} {tag:26s} AUROC {np.mean(a):.3f} +/- {np.std(a):.3f}")

# 2. how much of the data comes from each paper?
pp = (df.groupby("reference")
        .agg(rows=("label", "size"), synergy_rate=("label", "mean"),
             peptides=("sequence", "nunique"), antibiotics=("antibiotic", "nunique"))
        .sort_values("rows", ascending=False))
print("\n", pp.round(2).to_string())
print("rows with no reference number:", int(df["reference"].isna().sum()))

BASELINE  all rows                   AUROC 0.643 +/- 0.005
BASELINE  without polymyxin-B rows   AUROC 0.642 +/- 0.009
FULL      all rows                   AUROC 0.586 +/- 0.010
FULL      without polymyxin-B rows   AUROC 0.552 +/- 0.013

            rows  synergy_rate  peptides  antibiotics
reference                                           
1.0        1395          0.50       300           61
2.0         374          0.44        57           49
3.0         184          0.50        32           38
6.0          64          0.27         7           24
8.0          50          0.58         6           16
4.0          45          0.56         8           20
9.0          45          0.22         4           11
7.0          35          0.31         4           23
10.0         22          0.36         1            7
15.0         19          0.16         1            5
26.0         14          0.07         1            5
23.0         13          0.38         1           13
5.0          12     

## 9. What drives the baseline? Paper 1 versus the rest
SHAP shows which antibiotic/bacterium features the baseline relies on. The paper split checks whether the picture holds inside the dominant study and outside it.

In [ ]:
import shap

# 1. what drives the BASELINE model (antibiotic + bacterium only)?
m_base = make(0).fit(X_base, y)
sv = shap.TreeExplainer(m_base).shap_values(X_base)
names_b = (["abx_logP", "abx_MW", "abx_TPSA", "abx_HBD", "abx_HBA"]
           + [f"abx_fp{i}" for i in range(256)]
           + ["gram_neg"] + [f"species_{s}" for s in species_cols])
assert len(names_b) == X_base.shape[1]
imp = pd.Series(np.abs(sv).mean(0), index=names_b).sort_values(ascending=False)
print("top baseline features by mean |SHAP|:")
print(imp.head(15).round(3).to_string())

# 2. paper 1 vs all other papers (5 seeds each)
ref = df["reference"]
subsets = {"paper 1 only": (ref == 1).values,
           "all other papers": ((ref != 1) & ref.notna()).values}
sets = {"BASELINE": X_base,
        "BASE + charge,hmoment,length": np.hstack([X_pep[:, [1, 5, 0]], X_base]),
        "FULL": X_full}
for sname, m in subsets.items():
    print(f"\n{sname}: {m.sum()} rows, {df.loc[m, 'sequence'].nunique()} peptides")
    for name, X in sets.items():
        res = [evaluate(X, g_pep, StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=sd),
                        mask=m, seed=sd) for sd in range(5)]
        a = [r["AUROC"] for r in res]
        print(f"  {name:30s} AUROC {np.mean(a):.3f} +/- {np.std(a):.3f}")

top baseline features by mean |SHAP|:
abx_MW                                0.104
species_klebsiella pneumoniae         0.103
abx_fp164                             0.097
species_pseudomonas aeruginosa        0.088
abx_logP                              0.067
species_aeromonas veronii             0.061
species_staphylococcus aureus         0.055
abx_fp252                             0.054
abx_fp150                             0.053
abx_fp71                              0.050
species_staphylococcus epidermidis    0.047
species_streptococcus pneumoniae      0.045
abx_fp143                             0.041
abx_TPSA                              0.039
abx_fp40                              0.039

paper 1 only: 1395 rows, 300 peptides
  BASELINE                       AUROC 0.588 +/- 0.014
  BASE + charge,hmoment,length   AUROC 0.606 +/- 0.014
  FULL                           AUROC 0.565 +/- 0.016

all other papers: 923 rows, 126 peptides
  BASELINE                       AUROC 0.595 +/- 0.015
 

## 10. Environment
Library versions used for this run, recorded for reproducibility.

In [ ]:
import sys, platform
import numpy, pandas, sklearn, scipy, xgboost, rdkit, Bio, shap
try:
    import torch, transformers
    tv = (torch.__version__, transformers.__version__)
except Exception:
    tv = ("n/a", "n/a")
print("python", sys.version.split()[0], "|", platform.platform())
for name, mod in [("numpy", numpy), ("pandas", pandas), ("scikit-learn", sklearn), ("scipy", scipy),
                  ("xgboost", xgboost), ("rdkit", rdkit), ("biopython", Bio), ("shap", shap)]:
    print(f"{name:14s}", mod.__version__)
print("torch / transformers:", *tv)

python 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
numpy          2.4.6
pandas         2.2.3
scikit-learn   1.6.1
scipy          1.16.3
xgboost        3.4.1
rdkit          2026.09.2
biopython      1.88
shap           0.54.1
torch / transformers: 2.14.1+cu130 5.22.1


## Summary of results (this run)
* New peptide families (5 seeds): baseline AUROC **0.643 ± 0.005**; every variant with peptide features was lower (0.586–0.613).
* New antibiotic: full model 0.758, baseline 0.590, **peptide-only 0.767** → the apparent generalization is peptide memorization.
* New study: baseline 0.669, full 0.652, peptide-only 0.613. 96% of peptides occur in only one study.
* Dropping the polymyxin-B stand-in rows leaves the baseline unchanged (0.643 → 0.642); the full model drops (0.586 → 0.552).
* The data is dominated by one study (study 1: 1,395 of 2,325 rows).

Re-running can give small numeric differences (library versions, hardware); the conclusions should not change.